# Fraud Detection - Logistic Regression

This notebook builds and evaluates a Logistic Regression model for fraud detection.

The main goals are to:
- prepare the features for Logistic Regression,
- train a baseline classification model,
- evaluate its performance on the validation dataset,
- understand the limitations of accuracy in an imbalanced fraud detection problem.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
)

from finsight.database import (
    connect_to_database,
    test_database_connection,
)

from finsight.fraud_data import (
    download_train_data,
    download_validation_data,
)

from finsight.fraud_preprocessing import (
    encode_categorical_features,
    handle_missing_values,
)

## Load Modeling Data

The training and validation datasets are loaded from the PostgreSQL analytics layer using the reusable functions defined in the `finsight` package.

In [2]:
engine = connect_to_database()
test_database_connection(engine)

Connected!


In [3]:
X_train, y_train = download_train_data(engine)

In [4]:
X_val, y_val = download_validation_data(engine)

## Categorical Feature Encoding

Logistic Regression requires numerical input features. The feature types are inspected to identify any categorical variables that need to be converted into a numerical representation.

In [5]:
X_train.dtypes

amount                             float64
mcc_key                              int64
use_chip                               str
merchant_id                          int64
transaction_hour                   float64
day_of_week                        float64
is_weekend                           int64
user_previous_transaction_count      int64
user_previous_avg_amount           float64
amount_vs_user_avg                 float64
card_previous_transaction_count      int64
card_previous_avg_amount           float64
amount_vs_card_avg                 float64
dtype: object

In [6]:
X_train["use_chip"].value_counts(dropna=False)

use_chip
Swipe Transaction     183906
Chip Transaction       83080
Online Transaction     43357
Name: count, dtype: int64

The `use_chip` feature contains three transaction categories: chip, swipe and online transactions.

These categories do not have a natural numerical order, so assigning values such as 1, 2 and 3 would introduce an artificial relationship between them. One-hot encoding is therefore used to represent each category as a separate binary feature.

The encoder is fitted only on the training data and the same fitted encoder is then used to transform the validation data.

In [7]:
final_train, final_val = encode_categorical_features(X_train, X_val)

## Missing Values

Before training the model, the encoded datasets are checked for missing values.

In [8]:
final_train.isna().sum()

amount                              0
mcc_key                             0
merchant_id                         0
transaction_hour                    0
day_of_week                         0
is_weekend                          0
user_previous_transaction_count     0
user_previous_avg_amount           30
amount_vs_user_avg                 30
card_previous_transaction_count     0
card_previous_avg_amount           93
amount_vs_card_avg                 93
use_chip_Chip Transaction           0
use_chip_Online Transaction         0
use_chip_Swipe Transaction          0
dtype: int64

In [9]:
final_train, final_val = handle_missing_values(final_train, final_val)

In [10]:
final_train.isna().sum()

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

In [11]:
final_val.isna().sum()

amount                             0
mcc_key                            0
merchant_id                        0
transaction_hour                   0
day_of_week                        0
is_weekend                         0
user_previous_transaction_count    0
user_previous_avg_amount           0
amount_vs_user_avg                 0
card_previous_transaction_count    0
card_previous_avg_amount           0
amount_vs_card_avg                 0
use_chip_Chip Transaction          0
use_chip_Online Transaction        0
use_chip_Swipe Transaction         0
dtype: int64

## Baseline Logistic Regression

With the categorical feature encoded and missing values handled, a first Logistic Regression model can be trained.

This model serves as the initial baseline before investigating its performance in more detail.

In [12]:
baseline_model = LogisticRegression()
baseline_model.fit(final_train, y_train)
print("Training set score: {:.2f}".format(baseline_model.score(final_train, y_train)))
print("Validation set score: {:.2f}".format(baseline_model.score(final_val, y_val)))

Training set score: 0.97
Validation set score: 1.00


/Users/martyna/Documents/Projects/finsight-fintech-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
y_pred = baseline_model.predict(final_val)

In [14]:
y_pred.sum()

np.int64(557)

## Model Evaluation

Accuracy can be misleading in fraud detection because the validation dataset is highly imbalanced. Therefore, the model is evaluated using precision, recall, F1-score and the confusion matrix.

### Recall

In [15]:
recall = recall_score(y_val, y_pred)
print("Recall: {:.3f}".format(recall))

Recall: 0.004


### Precision

In [16]:
precision = precision_score(y_val, y_pred)
print("Precision: {:.3f}".format(precision))

Precision: 0.013


### F1

In [17]:
f1 = f1_score(y_val, y_pred)
print("F1: {:.3f}".format(f1))

F1: 0.006


### Confusion matrix

In [18]:
conf_matrix = confusion_matrix(y_val, y_pred)
print("Confusion matrix")
print(conf_matrix)

Confusion matrix
[[932420    550]
 [  1622      7]]


### Summary

Despite the almost perfect validation accuracy, a deeper analysis using recall, precision and F1-score shows that the model performs poorly at detecting fraud.

The model correctly classifies most transactions mainly because fraud represents only about 0.17% of the validation dataset. As a result, high accuracy does not reflect the model's actual ability to identify fraudulent transactions.

## Model Convergence

During training, Logistic Regression returned a convergence warning, indicating that the optimization algorithm did not reach a stable solution within the default number of iterations.

Before modifying the model, the cause of this warning should be investigated.

In [19]:
lrg = LogisticRegression(max_iter = 1000)
lrg.fit(final_train, y_train)
print("Training set score: {:.2f}".format(lrg.score(final_train, y_train)))
print("Validation set score: {:.2f}".format(lrg.score(final_val, y_val)))

Training set score: 0.97
Validation set score: 1.00


/Users/martyna/Documents/Projects/finsight-fintech-analytics/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Increasing the maximum number of iterations from 100 to 1,000 did not resolve the convergence warning. This suggests that the issue is not caused solely by an insufficient iteration limit.

Since the numerical features operate on substantially different scales, feature scaling will be investigated next.

### Feature Scaling

The convergence warning persisted even after increasing the maximum number of iterations. Since the numerical features operate on different scales, StandardScaler is applied before training the model again.

The scaler is fitted only on the training data and the same transformation is then applied to the validation data.

In [20]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(final_train)

scaled_data_val = scaler.transform(final_val)

## Logistic Regression with Scaled Features

A new Logistic Regression model is trained using the scaled features to determine whether feature scaling resolves the convergence problem.

In [21]:
scaled_model = LogisticRegression()
scaled_model.fit(scaled_data, y_train)
print("Training set score: {:.2f}".format(scaled_model.score(scaled_data, y_train)))
print("Validation set score: {:.2f}".format(scaled_model.score(scaled_data_val, y_val)))

Training set score: 0.97
Validation set score: 1.00


Feature scaling resolved the convergence issue, as the model now trains without a convergence warning.

However, the training and validation accuracy remain almost unchanged. Since accuracy is misleading for this highly imbalanced dataset, the scaled model must be evaluated using fraud-focused metrics.

In [22]:
y_pred_scaled = scaled_model.predict(scaled_data_val)

## Model Evaluation

### Recall

In [23]:
recall_scaled = recall_score(y_val, y_pred_scaled)
print("Recall: {:.3f}".format(recall_scaled))

Recall: 0.009


### Precision

In [24]:
precision_scaled = precision_score(y_val, y_pred_scaled)
print("Precision: {:.3f}".format(precision_scaled))

Precision: 0.010


### F1

In [25]:
f1_scaled = f1_score(y_val, y_pred_scaled)
print("F1: {:.3f}".format(f1_scaled))

F1: 0.010


### Confusion Matrix

In [26]:
conf_matrix_scaled = confusion_matrix(y_val, y_pred_scaled)
print("Confusion Matrix")
print(conf_matrix_scaled)

Confusion Matrix
[[931527   1443]
 [  1614     15]]


### Baseline vs Scaled Model

The baseline and scaled models are compared using fraud-focused evaluation metrics to assess whether feature scaling improved predictive performance in addition to resolving the convergence issue.

In [27]:
model_comparison = {
    "Metric": ["Recall", "Precision", "F1"],
    "Baseline": [recall, precision, f1],
     "Scaled": [recall_scaled, precision_scaled, f1_scaled]
}

model_comparison = pd.DataFrame(model_comparison).set_index("Metric")
model_comparison

,Baseline,Scaled
Metric,,
Recall,0.004297,0.009208
Precision,0.012567,0.010288
F1,0.006404,0.009718


### Comparison Summary

Feature scaling resolved the convergence problem and slightly improved recall, precision and F1-score compared with the baseline model.

However, the improvement is small and the overall fraud detection performance remains poor. The scaled model still identifies less than 1% of fraudulent transactions, showing that convergence was not the main reason for the model's weak predictive performance.

## Predicted Fraud Probabilities

Although feature scaling resolved the convergence issue, the model still detects very few fraudulent transactions.

Before testing another model, the predicted probabilities are examined to understand how Logistic Regression separates fraud and non-fraud transactions before the default classification threshold is applied.

In [28]:
y_proba_scaled = scaled_model.predict_proba(scaled_data_val)
y_proba_scaled

array([[0.99768986, 0.00231014],
       [0.99279607, 0.00720393],
       [0.99279948, 0.00720052],
       ...,
       [0.98858725, 0.01141275],
       [0.99337709, 0.00662291],
       [0.99304039, 0.00695961]], shape=(934599, 2))

In [29]:
fraud_proba = y_proba_scaled[:, 1]

In [30]:
print("Min probability for fraud: {:.2f}".format(min(fraud_proba)))
print("Max probability for fraud: {:.2f}".format(max(fraud_proba)))
print("Avg probability for fraud: {:.2f}".format(fraud_proba.mean()))

Min probability for fraud: 0.00
Max probability for fraud: 0.99
Avg probability for fraud: 0.02


In [31]:
fraud_proba_true = fraud_proba[y_val == 1]
mean_fraud_proba_true = fraud_proba_true.mean()
median_fraud_proba_true = np.median(fraud_proba_true)
q1_fraud_proba_true = np.quantile(fraud_proba_true, 0.25)
q3_fraud_proba_true = np.quantile(fraud_proba_true, 0.75)

In [32]:
fraud_proba_false = fraud_proba[y_val == 0]
mean_fraud_proba_false = fraud_proba_false.mean()
median_fraud_proba_false = np.median(fraud_proba_false)
q1_fraud_proba_false = np.quantile(fraud_proba_false, 0.25)
q3_fraud_proba_false = np.quantile(fraud_proba_false, 0.75)

In [33]:
probability_summary = {
    "Is_Fraud": ["True fraud", "Non-fraud"],
    "Mean": [mean_fraud_proba_true, mean_fraud_proba_false],
    "Q1": [q1_fraud_proba_true, q1_fraud_proba_false],
    "Median": [median_fraud_proba_true, median_fraud_proba_false],
    "Q3": [q3_fraud_proba_true, q3_fraud_proba_false]
}

probability_summary = pd.DataFrame(probability_summary).set_index("Is_Fraud")
probability_summary

,Mean,Q1,Median,Q3
Is_Fraud,,,,
True fraud,0.024649,0.003318,0.005893,0.012813
Non-fraud,0.017930,0.001953,0.003779,0.007693


#### Probability Distribution Summary

Actual fraudulent transactions receive higher predicted fraud probabilities across the mean, median and quartiles than non-fraudulent transactions. This indicates that the model captures some signal that helps distinguish between the two classes.

However, the probability distributions still overlap substantially and most predicted probabilities remain far below the default classification threshold of 0.5.

Therefore, the next step is to investigate how changing the classification threshold affects recall and precision.

## Classification Threshold Analysis

Logistic Regression converts predicted probabilities into class labels using a default classification threshold of 0.5.

The probability analysis showed that most transactions receive fraud probabilities well below this threshold, including transactions that are actually fraudulent.

The next step is to investigate how lowering the classification threshold affects the model's ability to detect fraud and the trade-off between recall and precision.

In [34]:
thresholds = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (fraud_proba >= threshold).astype(int)

    recall_threshold = recall_score(y_val, y_pred_threshold)
    precision_threshold = precision_score(y_val, y_pred_threshold)
    f1_threshold = f1_score(y_val, y_pred_threshold)

    threshold_results.append({
        "Threshold": threshold,
        "Recall": recall_threshold,
        "Precision": precision_threshold,
        "F1": f1_threshold
    })

threshold_results = pd.DataFrame(threshold_results).set_index("Threshold")
threshold_results

,Recall,Precision,F1
Threshold,,,
0.50,0.009208,0.010288,0.009718
0.20,0.023327,0.002225,0.004062
0.10,0.046041,0.001686,0.003253
0.05,0.076734,0.001545,0.003029
0.02,0.182320,0.002419,0.004775
0.01,0.309392,0.002732,0.005415


#### Threshold Analysis Summary

Lowering the classification threshold substantially increases recall, allowing the model to identify a larger proportion of fraudulent transactions. At a threshold of 0.01, recall increases from less than 1% to approximately 31%.

However, this improvement comes at the cost of very low precision. Even at lower thresholds, only a small proportion of transactions classified as fraud are actually fraudulent, resulting in a large number of false positives.

The experiment demonstrates the trade-off between recall and precision, but also shows that threshold adjustment alone is not sufficient to achieve strong fraud detection performance. Further improvements should therefore focus on the modeling approach rather than continuing to lower the decision threshold.